# 03 Model Training and Evaluation

Train and evaluate Malaysia state-level poverty models from the retained model-ready state panel. All preprocessing is fitted inside each validation fold. The primary model is chosen by a fixed, sensor-safe rule and is assessed with both Leave-One-State-Out validation and a rolling-origin future-year robustness check. Full-feature models that use the 2013 VIIRS substitute for 2012 are retained only as retrospective comparators.


In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import pearsonr, spearmanr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = Path.cwd().parent

QA_DIR = PROJECT_ROOT / "outputs" / "qa"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
INTERMEDIATE_DIR = PROJECT_ROOT / "outputs" / "intermediate"
for path in [QA_DIR, METRICS_DIR, INTERMEDIATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Self-contained benchmark helpers. These are inlined so this repository notebook
# can run without the development-only helper package.
from dataclasses import dataclass
from typing import Callable

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


MetricFunc = Callable[[np.ndarray, np.ndarray], float]


@dataclass(frozen=True)
class MetricSpec:
    name: str
    func: MetricFunc
    higher_is_better: bool


def _safe_corr(kind: str, y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2 or len(np.unique(y_pred)) < 2:
        return float("nan")
    if kind == "spearman":
        return float(spearmanr(y_true, y_pred).statistic)
    return float(pearsonr(y_true, y_pred).statistic)


METRICS: dict[str, MetricSpec] = {
    "mae": MetricSpec("mae", lambda y, p: float(mean_absolute_error(y, p)), False),
    "rmse": MetricSpec("rmse", lambda y, p: float(mean_squared_error(y, p) ** 0.5), False),
    "r2": MetricSpec("r2", lambda y, p: float(r2_score(y, p)), True),
    "spearman": MetricSpec("spearman", lambda y, p: _safe_corr("spearman", y, p), True),
    "pearson": MetricSpec("pearson", lambda y, p: _safe_corr("pearson", y, p), True),
}


def summarize_metrics(y_true: pd.Series | np.ndarray, y_pred: pd.Series | np.ndarray) -> dict[str, float]:
    """Return the standard state-model metric set used in audit artifacts."""
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    return {name: spec.func(y, p) for name, spec in METRICS.items()}


def grouped_bootstrap_ci(
    frame: pd.DataFrame,
    *,
    y_col: str,
    pred_col: str,
    group_col: str,
    metric_names: list[str] | None = None,
    n_boot: int = 1000,
    random_state: int = 42,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """Bootstrap held-out groups while preserving all rows within each sampled group."""
    metric_names = metric_names or ["mae", "rmse", "r2", "spearman", "pearson"]
    groups = np.asarray(sorted(frame[group_col].dropna().unique()))
    group_indices = {
        group: np.flatnonzero(frame[group_col].to_numpy() == group) for group in groups
    }
    rng = np.random.default_rng(random_state)
    observed = {
        name: METRICS[name].func(frame[y_col].to_numpy(float), frame[pred_col].to_numpy(float))
        for name in metric_names
    }
    samples: dict[str, list[float]] = {name: [] for name in metric_names}
    for _ in range(n_boot):
        sampled_groups = rng.choice(groups, size=len(groups), replace=True)
        indices = np.concatenate([group_indices[group] for group in sampled_groups])
        y = frame.iloc[indices][y_col].to_numpy(float)
        pred = frame.iloc[indices][pred_col].to_numpy(float)
        for name in metric_names:
            try:
                value = METRICS[name].func(y, pred)
            except ValueError:
                value = float("nan")
            if np.isfinite(value):
                samples[name].append(float(value))
    rows = []
    for name in metric_names:
        lower, upper = np.nanpercentile(
            samples[name], [100 * alpha / 2, 100 * (1 - alpha / 2)]
        )
        rows.append(
            {
                "metric": name,
                "observed": observed[name],
                "ci_lower": float(lower),
                "ci_upper": float(upper),
                "n_boot": int(len(samples[name])),
                "group_col": group_col,
                "n_groups": int(len(groups)),
            }
        )
    return pd.DataFrame(rows)


def paired_grouped_delta_ci(
    frame: pd.DataFrame,
    *,
    y_col: str,
    candidate_col: str,
    baseline_col: str,
    group_col: str,
    metric_name: str = "mae",
    n_boot: int = 1000,
    random_state: int = 42,
    alpha: float = 0.05,
) -> dict[str, float | str | int]:
    """Estimate paired candidate-vs-baseline improvement by resampling groups."""
    spec = METRICS[metric_name]
    groups = np.asarray(sorted(frame[group_col].dropna().unique()))
    group_indices = {
        group: np.flatnonzero(frame[group_col].to_numpy() == group) for group in groups
    }
    rng = np.random.default_rng(random_state)

    def improvement(indices: np.ndarray) -> float:
        sample = frame.iloc[indices]
        y = sample[y_col].to_numpy(float)
        candidate = spec.func(y, sample[candidate_col].to_numpy(float))
        baseline = spec.func(y, sample[baseline_col].to_numpy(float))
        return float(candidate - baseline if spec.higher_is_better else baseline - candidate)

    all_indices = np.arange(len(frame))
    observed = improvement(all_indices)
    samples = []
    for _ in range(n_boot):
        sampled_groups = rng.choice(groups, size=len(groups), replace=True)
        indices = np.concatenate([group_indices[group] for group in sampled_groups])
        value = improvement(indices)
        if np.isfinite(value):
            samples.append(float(value))
    lower, upper = np.nanpercentile(samples, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {
        "metric": metric_name,
        "candidate_col": candidate_col,
        "baseline_col": baseline_col,
        "observed_improvement": float(observed),
        "ci_lower": float(lower),
        "ci_upper": float(upper),
        "n_boot": int(len(samples)),
        "group_col": group_col,
        "n_groups": int(len(groups)),
    }



from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

TARGETS = ["poverty_absolute", "poverty_relative"]

XGBOOST_PARAMS = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": 1,
    "tree_method": "hist",
}

RANDOM_FOREST_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "min_samples_leaf": 2,
    "random_state": 42,
    "n_jobs": 1,
}

NON_FEATURE_COLS = {
    "state_key",
    "state",
    "poverty_absolute",
    "poverty_hardcore",
    "poverty_relative",
    "mean_target_year",
    "wmean_target_year",
    "mean_ntl_sensor_code",
    "wmean_ntl_sensor_code",
    "mean_ntl_year_used",
    "mean_pop_year_used",
    "wmean_ntl_year_used",
    "wmean_pop_year_used",
    "mean_requested_year",
    "mean_land_cover_year_used",
    "mean_worldpop_year_used",
    "mean_boundary_version_code",
}

PROVENANCE_MODEL_EXCLUDE_COLUMNS = {
    "requested_year",
    "sat_year_used",
    "land_cover_year_used",
    "worldpop_year_used",
    "boundary_version",
    "ntl_sensor",
}
DERIVED_SENSOR_FEATURES = {
    "sensor_era_viirs",
    "year_centered",
    "viirs_year_interaction",
}

RAW_NTL_TOKENS = ("ntl_mean", "ntl_median", "ntl_highshare")
TRANSLECTIVE_YEAR_SUFFIXES = ("_z_by_year", "_rank_by_year")
STATIC_CONTEXT_TOKENS = (
    "elevation",
    "slope",
    "frac_urban",
    "frac_cropland",
    "frac_forest",
    "pop_sum",
    "population_density",
)


@dataclass(frozen=True)
class ModelRun:
    name: str
    target: str
    estimator: object
    feature_policy: str
    feature_cols: list[str]


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def write_json(path: Path, payload: dict) -> dict:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    return payload


def drop_correlated_features(X: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [col for col in upper.columns if any(upper[col] > threshold)]
    return X.drop(columns=drop_cols)


def prepare_state_panel(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = {"state_key", "state", "year", *TARGETS} - set(df.columns)
    if missing:
        raise ValueError(f"State panel missing required columns: {sorted(missing)}")
    df = df.sort_values(["state_key", "year"]).reset_index(drop=True)
    df["year"] = df["year"].astype(int)
    df["sensor_era_viirs"] = (df.get("wmean_ntl_sensor_code", 0).fillna(0) >= 1).astype(int)
    df["year_centered"] = df["year"] - df["year"].median()
    df["viirs_year_interaction"] = df["sensor_era_viirs"] * df["year_centered"]
    return df


def eligible_numeric_features(df: pd.DataFrame) -> list[str]:
    excluded = NON_FEATURE_COLS | PROVENANCE_MODEL_EXCLUDE_COLUMNS
    cols = [
        c
        for c in df.select_dtypes(include=[np.number]).columns
        if c not in excluded
    ]
    return cols


def feature_sets(df: pd.DataFrame) -> dict[str, list[str]]:
    """Return fixed, predeclared feature policies without inspecting target values."""
    current_full = [
        "year", "pop_sum_grid_log1p", "wmean_elevation", "wmean_evi_mean",
        "wmean_frac_cropland", "wmean_frac_forest", "wmean_frac_urban",
        "wmean_ndvi_peak", "wmean_ndvi_range", "wmean_ntl_highshare",
        "wmean_ntl_mean", "wmean_slope", "dist_ntl_mean_mean",
        "dist_ntl_mean_std", "dist_ntl_mean_p10", "dist_ntl_mean_iqr",
        "dist_ntl_mean_cv", "dist_urban_cell_share_thr02",
        "dist_bright_cell_share_thr1", "dist_bright_pop_share_thr1",
    ]
    sensor_safe = [
        "year", "pop_sum_grid_log1p", "wmean_elevation", "wmean_evi_mean",
        "wmean_frac_cropland", "wmean_frac_forest", "wmean_frac_urban",
        "wmean_ndvi_peak", "wmean_ndvi_range", "wmean_slope",
        "dist_urban_cell_share_thr02",
    ]
    required = set(current_full) | set(sensor_safe)
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Model-ready panel is missing declared features: {sorted(missing)}")
    return {"year_only": ["year"], "current_full": current_full, "sensor_safe": sensor_safe}


def _tree_pipeline(model: object) -> Pipeline:
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", model),
    ])


def _xgb() -> Pipeline:
    return _tree_pipeline(XGBRegressor(**XGBOOST_PARAMS))


def _rf() -> Pipeline:
    return _tree_pipeline(RandomForestRegressor(**RANDOM_FOREST_PARAMS))


def _scaled_linear(model: object) -> Pipeline:
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", model),
    ])


def _year_dummies_ridge(numeric_cols: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        [
            ("year", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), ["year"]),
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), [c for c in numeric_cols if c != "year"]),
        ],
        remainder="drop",
    )
    return Pipeline([("prep", preprocessor), ("model", Ridge(alpha=3.0))])


def build_model_runs(target: str, features: dict[str, list[str]]) -> list[ModelRun]:
    return [
        ModelRun("year_only_xgb", target, _xgb(), "year_only", features["year_only"]),
        ModelRun("current_xgb", target, _xgb(), "current_full", features["current_full"]),
        ModelRun("sensor_safe_xgb", target, _xgb(), "sensor_safe", features["sensor_safe"]),
        ModelRun("rf_current_full", target, _rf(), "current_full", features["current_full"]),
        ModelRun("rf_sensor_safe", target, _rf(), "sensor_safe", features["sensor_safe"]),
        ModelRun("sensor_safe_ridge", target, _scaled_linear(Ridge(alpha=3.0)), "sensor_safe", features["sensor_safe"]),
        ModelRun(
            "sensor_safe_elasticnet",
            target,
            _scaled_linear(ElasticNet(alpha=0.05, l1_ratio=0.2, max_iter=20000, random_state=42)),
            "sensor_safe",
            features["sensor_safe"],
        ),
        ModelRun(
            "sensor_safe_year_dummies_ridge",
            target,
            _year_dummies_ridge(features["sensor_safe"]),
            "sensor_safe_year_dummies",
            features["sensor_safe"],
        ),
    ]


def loso_predictions(df: pd.DataFrame, run: ModelRun) -> pd.DataFrame:
    model_df = df.dropna(subset=[run.target]).copy()
    X = model_df[run.feature_cols].copy()
    y = model_df[run.target].astype(float)
    y_log = np.log1p(y)
    groups = model_df["state_key"]
    logo = LeaveOneGroupOut()
    preds_log = cross_val_predict(clone(run.estimator), X, y_log, groups=groups, cv=logo, n_jobs=None)
    preds = np.expm1(preds_log).clip(0)
    out = model_df[["state_key", "state", "year"]].copy()
    out["target"] = run.target
    out["model"] = run.name
    out["feature_policy"] = run.feature_policy
    out["actual"] = y.to_numpy()
    out["predicted"] = preds
    out["residual"] = out["predicted"] - out["actual"]
    out["abs_error"] = out["residual"].abs()
    out["feature_count"] = len(X.columns)
    return out


def fit_full_predictions(df: pd.DataFrame, run: ModelRun) -> np.ndarray:
    model_df = df.dropna(subset=[run.target]).copy()
    X = model_df[run.feature_cols].copy()
    y = model_df[run.target].astype(float)
    model = clone(run.estimator)
    model.fit(X, np.log1p(y))
    return np.expm1(model.predict(X)).clip(0)


def rolling_origin_predictions(
    df: pd.DataFrame, run: ModelRun, *, min_train_years: int = 4
) -> pd.DataFrame:
    """Predict each later survey year using only strictly earlier years."""
    model_df = df.dropna(subset=[run.target]).copy()
    years = sorted(int(year) for year in model_df["year"].unique())
    rows: list[pd.DataFrame] = []
    for test_year in years[min_train_years:]:
        train = model_df[model_df["year"] < test_year]
        test = model_df[model_df["year"] == test_year]
        if train["year"].nunique() < min_train_years or test.empty:
            continue
        model = clone(run.estimator)
        model.fit(train[run.feature_cols], np.log1p(train[run.target].astype(float)))
        predicted = np.expm1(model.predict(test[run.feature_cols])).clip(0)
        fold = test[["state_key", "state", "year"]].copy()
        fold["target"] = run.target
        fold["model"] = run.name
        fold["train_end_year"] = int(train["year"].max())
        fold["actual"] = test[run.target].to_numpy(float)
        fold["predicted"] = predicted
        fold["residual"] = fold["predicted"] - fold["actual"]
        fold["abs_error"] = fold["residual"].abs()
        rows.append(fold)
    if not rows:
        raise ValueError("Rolling-origin evaluation produced no test folds.")
    return pd.concat(rows, ignore_index=True)


def _feature_group(feature: str) -> str:
    lower = feature.lower()
    if lower == "year":
        return "year/time"
    if "ntl" in lower or "bright" in lower:
        return "nighttime lights"
    if "year" in lower:
        return "year/time"
    if "ndvi" in lower or "evi" in lower or "vegetation" in lower:
        return "vegetation"
    if "urban" in lower:
        return "urbanisation"
    if "pop" in lower:
        return "population"
    if "slope" in lower or "elevation" in lower:
        return "topography"
    if "cropland" in lower or "forest" in lower:
        return "land cover"
    return "other"


def _rf_shap_agreement_flags(features: list[str]) -> dict[str, bool]:
    feature_set = set(features)
    return {
        "time_trend": "year" in feature_set,
        "vegetation": bool(feature_set & {"wmean_ndvi_peak", "wmean_evi_mean"}),
        "nighttime_lights": bool(
            feature_set
            & {
                "wmean_ntl_mean",
                "wmean_ntl_highshare",
                "wmean_ntl_mean_z_by_year",
                "wmean_ntl_highshare_rank_by_year",
            }
        ),
        "urbanisation": bool(feature_set & {"wmean_frac_urban", "dist_urban_cell_share_thr02"}),
    }


def generate_rf_shap_artifacts(
    df: pd.DataFrame,
    run: ModelRun,
    *,
    project_root: Path,
    metrics_dir: Path,
    qa_dir: Path,
) -> dict[str, object]:
    """Fit selected RF on the full panel and write descriptive TreeSHAP artifacts."""
    if run.name != "rf_sensor_safe" or run.target != "poverty_absolute":
        raise ValueError("RF SHAP is only generated for the selected sensor-safe RF.")

    figures_dir = project_root / "outputs" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

    model_df = df.dropna(subset=[run.target]).copy()
    X = model_df[run.feature_cols].copy()
    y = model_df[run.target].astype(float)

    model = clone(run.estimator)
    model.fit(X, np.log1p(y))
    X_explain = pd.DataFrame(
        model.named_steps["impute"].transform(X), columns=X.columns, index=X.index
    )
    rf_model = model.named_steps["model"]

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import shap

    explainer = shap.TreeExplainer(rf_model)
    shap_values = np.asarray(explainer.shap_values(X_explain))
    mean_abs = np.abs(shap_values).mean(axis=0)
    mean_signed = shap_values.mean(axis=0)

    importance = pd.DataFrame(
        {
            "feature": X_explain.columns,
            "feature_group": [_feature_group(col) for col in X_explain.columns],
            "mean_abs_shap_log_points": mean_abs,
            "mean_signed_shap_log_points": mean_signed,
        }
    ).sort_values("mean_abs_shap_log_points", ascending=False)
    importance["mean_abs_rank"] = range(1, len(importance) + 1)
    total = float(importance["mean_abs_shap_log_points"].sum())
    importance["importance_share"] = (
        importance["mean_abs_shap_log_points"] / total if total else 0.0
    )

    table_path = metrics_dir / "malaysia_state_rf_shap_poverty_absolute.csv"
    importance.to_csv(table_path, index=False)

    figure_path = figures_dir / "mys_state_rf_shap_poverty_absolute.png"
    plt.figure(figsize=(9, 6.8))
    shap.summary_plot(shap_values, X_explain, show=False, max_display=20)
    plt.title("Random Forest SHAP - Malaysia State Absolute Poverty", fontsize=12)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.close()

    grouped = (
        importance.groupby("feature_group", as_index=False)["mean_abs_shap_log_points"]
        .sum()
        .sort_values("mean_abs_shap_log_points", ascending=False)
    )
    grouped["importance_share"] = (
        grouped["mean_abs_shap_log_points"] / float(grouped["mean_abs_shap_log_points"].sum())
    )

    top_features = importance.head(12)
    summary = {
        "target": run.target,
        "model": run.name,
        "feature_policy": run.feature_policy,
        "feature_count": int(len(X_explain.columns)),
        "panel_rows": int(len(X)),
        "shap_scale": "log1p poverty percentage points; descriptive post-selection explanation only",
        "validation_caveat": (
            "RF SHAP is computed after model selection on the full Malaysia state-level panel. "
            "It explains fitted model behavior and is not held-out validation or causal evidence."
        ),
        "agreement_with_xgboost_story": _rf_shap_agreement_flags(top_features["feature"].tolist()),
        "top_features": top_features[
            [
                "mean_abs_rank",
                "feature",
                "feature_group",
                "mean_abs_shap_log_points",
                "importance_share",
            ]
        ].round(6).to_dict(orient="records"),
        "grouped_importance": grouped.round(6).to_dict(orient="records"),
        "figure_path": str(figure_path.relative_to(project_root)),
        "table_path": str(table_path.relative_to(project_root)),
    }
    summary_path = qa_dir / "malaysia_state_rf_shap_summary.json"
    summary["summary_path"] = str(summary_path.relative_to(project_root))
    write_json(summary_path, summary)
    return summary


def run_state_benchmark(
    *,
    project_root: Path = PROJECT_ROOT,
    n_boot: int = 1000,
    random_state: int = 42,
) -> dict[str, object]:
    metrics_dir = project_root / "outputs" / "metrics"
    qa_dir = project_root / "outputs" / "qa"
    metrics_dir.mkdir(parents=True, exist_ok=True)
    qa_dir.mkdir(parents=True, exist_ok=True)

    panel_path = project_root / "data" / "state_year_panel_modelready_2002_2022.csv"
    model_predictions_path = project_root / "data" / "state_model_predictions.csv"
    current_metrics_path = project_root / "outputs" / "metrics" / "malaysia_state_selected_model_loso_metrics.json"

    df = prepare_state_panel(panel_path)
    features = feature_sets(df)
    current_metrics = read_json(current_metrics_path)
    prediction_frames: list[pd.DataFrame] = []
    comparison_rows: list[dict[str, object]] = []

    for target in TARGETS:
        for run in build_model_runs(target, features):
            pred = loso_predictions(df, run)
            prediction_frames.append(pred)
            metrics = summarize_metrics(pred["actual"], pred["predicted"])
            comparison_rows.append(
                {
                    "target": target,
                    "model": run.name,
                    "feature_policy": run.feature_policy,
                    "feature_count": int(pred["feature_count"].iloc[0]),
                    "loso_mae_pp": round(metrics["mae"], 4),
                    "loso_rmse": round(metrics["rmse"], 4),
                    "loso_r2": round(metrics["r2"], 4),
                    "loso_spearman": round(metrics["spearman"], 4),
                    "loso_pearson": round(metrics["pearson"], 4),
                    "n_state_years": int(len(pred)),
                    "n_states": int(pred["state_key"].nunique()),
                    "n_survey_years": int(pred["year"].nunique()),
                }
            )

    predictions = pd.concat(prediction_frames, ignore_index=True)
    comparison = pd.DataFrame(comparison_rows).sort_values(["target", "loso_mae_pp", "model"]).reset_index(drop=True)

    # Primary selection is deterministic and restricted to the predeclared sensor-safe policy.
    absolute = comparison[comparison["target"] == "poverty_absolute"].copy()
    current_row = absolute[absolute["model"] == "current_xgb"].iloc[0]
    best_row = absolute.sort_values(["loso_mae_pp", "loso_spearman", "model"], ascending=[True, False, True]).iloc[0]
    eligible_absolute = absolute[absolute["feature_policy"] == "sensor_safe"].copy()
    selected_row = eligible_absolute.sort_values(
        ["loso_mae_pp", "loso_spearman", "model"], ascending=[True, False, True]
    ).iloc[0]
    selected_primary_model = str(selected_row["model"])
    relative_selected_model = str(
        comparison[comparison["target"] == "poverty_relative"]
        .sort_values(["loso_mae_pp", "loso_spearman", "model"], ascending=[True, False, True])
        .iloc[0]["model"]
    )
    current_pred = predictions[(predictions["target"] == "poverty_absolute") & (predictions["model"] == "current_xgb")]
    selected_pred = predictions[
        (predictions["target"] == "poverty_absolute") & (predictions["model"] == selected_primary_model)
    ]
    merged_abs = current_pred[["state_key", "state", "year", "actual", "predicted"]].rename(
        columns={"predicted": "current_xgb"}
    ).merge(
        selected_pred[["state_key", "year", "predicted"]].rename(columns={"predicted": selected_primary_model}),
        on=["state_key", "year"],
        how="inner",
    )
    delta_mae = paired_grouped_delta_ci(
        merged_abs,
        y_col="actual",
        candidate_col=selected_primary_model,
        baseline_col="current_xgb",
        group_col="state_key",
        metric_name="mae",
        n_boot=n_boot,
        random_state=random_state,
    )
    best_model = str(best_row["model"])
    best_pred = predictions[
        (predictions["target"] == "poverty_absolute") & (predictions["model"] == best_model)
    ]
    merged_best_abs = current_pred[["state_key", "state", "year", "actual", "predicted"]].rename(
        columns={"predicted": "current_xgb"}
    ).merge(
        best_pred[["state_key", "year", "predicted"]].rename(columns={"predicted": best_model}),
        on=["state_key", "year"],
        how="inner",
    )
    delta_mae_best = paired_grouped_delta_ci(
        merged_best_abs,
        y_col="actual",
        candidate_col=best_model,
        baseline_col="current_xgb",
        group_col="state_key",
        metric_name="mae",
        n_boot=n_boot,
        random_state=random_state,
    )
    promotion = {
        "promoted": True,
        "selected_model": selected_primary_model,
        "relative_selected_model": relative_selected_model,
        "best_absolute_model": str(best_row["model"]),
        "current_absolute_model": "current_xgb",
        "rule": (
            "For absolute poverty, restrict eligibility to the predeclared sensor-safe feature policy, "
            "then select lowest LOSO MAE; break ties by higher Spearman correlation and model name. "
            "Relative poverty uses the lowest-MAE benchmark and remains a secondary negative result."
        ),
        "manual_promotion_required": False,
        "manual_promotion_review_status": "not_applicable_deterministic_rule",
        "manual_promotion_reason": "No manual override is permitted by the encoded selection rule.",
        "current_absolute": current_row.to_dict(),
        "selected_absolute": selected_row.to_dict(),
        "best_absolute": best_row.to_dict(),
        "paired_mae_delta_selected_vs_current": delta_mae,
        "paired_mae_delta_best_vs_current": delta_mae_best,
        "certified_metrics_refreshed_from_local_training": True,
        "relative_poverty_caveat": "Relative poverty remains secondary; year-only is stronger and should not be claimed as satellite-improved.",
    }
    uncertainty_frames: list[pd.DataFrame] = []
    for (target, model), group in predictions.groupby(["target", "model"]):
        ci = grouped_bootstrap_ci(
            group,
            y_col="actual",
            pred_col="predicted",
            group_col="state_key",
            n_boot=n_boot,
            random_state=random_state,
        )
        ci.insert(0, "target", target)
        ci.insert(1, "model", model)
        uncertainty_frames.append(ci)
    uncertainty = pd.concat(uncertainty_frames, ignore_index=True)

    selected_abs_run = next(
        run for run in build_model_runs("poverty_absolute", features)
        if run.name == selected_primary_model
    )
    temporal_predictions = rolling_origin_predictions(df, selected_abs_run)
    temporal_metrics = summarize_metrics(
        temporal_predictions["actual"], temporal_predictions["predicted"]
    )

    predictions_path = metrics_dir / "malaysia_state_loso_fold_predictions.csv"
    comparison_path = metrics_dir / "malaysia_state_model_comparison.csv"
    uncertainty_path = metrics_dir / "malaysia_state_uncertainty_intervals.csv"
    temporal_path = metrics_dir / "malaysia_state_temporal_backtest.csv"
    temporal_summary_path = qa_dir / "malaysia_state_temporal_backtest.json"
    promotion_path = qa_dir / "malaysia_state_model_promotion.json"
    summary_path = qa_dir / "malaysia_state_benchmark_summary.json"
    rf_note_path = qa_dir / "malaysia_state_rf_continuity_benchmark.md"

    predictions.to_csv(predictions_path, index=False)
    comparison.to_csv(comparison_path, index=False)
    uncertainty.to_csv(uncertainty_path, index=False)
    temporal_predictions.to_csv(temporal_path, index=False)
    write_json(temporal_summary_path, {
        "target": "poverty_absolute",
        "model": selected_primary_model,
        "protocol": "rolling origin; every test year is predicted using strictly earlier survey years",
        "first_test_year": int(temporal_predictions["year"].min()),
        "last_test_year": int(temporal_predictions["year"].max()),
        "n_test_rows": int(len(temporal_predictions)),
        "mae_pp": round(temporal_metrics["mae"], 4),
        "rmse": round(temporal_metrics["rmse"], 4),
        "r2": round(temporal_metrics["r2"], 4),
        "spearman": round(temporal_metrics["spearman"], 4),
        "caveat": "Secondary robustness check; the model family was selected using LOSO results on the same panel.",
    })
    write_json(promotion_path, promotion)
    rf_note_path.write_text(
        _rf_continuity_note(comparison, uncertainty, promotion),
        encoding="utf-8",
    )

    rf_shap_run = selected_abs_run
    rf_shap_summary = generate_rf_shap_artifacts(
        df,
        rf_shap_run,
        project_root=project_root,
        metrics_dir=metrics_dir,
        qa_dir=qa_dir,
    )

    # Refresh certified dashboard/report contract from the local training run.
    # Absolute can promote a better model; relative uses year_only_xgb because
    # the relative-poverty evidence is trend-driven rather than satellite-improved.
    selected_model = str(promotion["selected_model"])
    selected_relative_model = str(promotion["relative_selected_model"])
    selected_abs_predictions = predictions[
        (predictions["target"] == "poverty_absolute") & (predictions["model"] == selected_model)
    ]
    selected_rel_predictions = predictions[
        (predictions["target"] == "poverty_relative") & (predictions["model"] == selected_relative_model)
    ]
    model_predictions_artifact = df[["state_key", "year"]].copy()
    model_predictions_artifact = model_predictions_artifact.merge(
        selected_abs_predictions[["state_key", "year", "predicted"]].rename(
            columns={"predicted": "predicted_poverty_absolute"}
        ),
        on=["state_key", "year"],
        how="left",
    )
    model_predictions_artifact = model_predictions_artifact.merge(
        selected_rel_predictions[["state_key", "year", "predicted"]].rename(
            columns={"predicted": "predicted_poverty_relative"}
        ),
        on=["state_key", "year"],
        how="left",
    )
    model_predictions_artifact["predicted_poverty_absolute"] = model_predictions_artifact["predicted_poverty_absolute"].round(6)
    model_predictions_artifact["predicted_poverty_relative"] = model_predictions_artifact["predicted_poverty_relative"].round(6)
    model_predictions_artifact.to_csv(model_predictions_path, index=False)

    selected_abs_row = comparison[
        (comparison["target"] == "poverty_absolute") & (comparison["model"] == selected_model)
    ].iloc[0]
    selected_rel_row = comparison[
        (comparison["target"] == "poverty_relative") & (comparison["model"] == selected_relative_model)
    ].iloc[0]
    updated_metrics = {
        "country": "MYS",
        "targets": {
            "poverty_absolute": _row_to_macro_metric(selected_abs_row, comparison, "poverty_absolute"),
            "poverty_relative": _row_to_macro_metric(selected_rel_row, comparison, "poverty_relative"),
        },
    }
    write_json(current_metrics_path, updated_metrics)

    summary = {
        "panel_rows": int(len(df)),
        "state_count": int(df["state_key"].nunique()),
        "survey_years": sorted(int(year) for year in df["year"].unique()),
        "feature_sets": {name: cols for name, cols in features.items()},
        "certified_metrics_after_run": updated_metrics,
        "comparison_path": str(comparison_path.relative_to(project_root)),
        "predictions_path": str(predictions_path.relative_to(project_root)),
        "uncertainty_path": str(uncertainty_path.relative_to(project_root)),
        "temporal_backtest_path": str(temporal_path.relative_to(project_root)),
        "temporal_backtest_summary_path": str(temporal_summary_path.relative_to(project_root)),
        "promotion_path": str(promotion_path.relative_to(project_root)),
        "rf_continuity_note_path": str(rf_note_path.relative_to(project_root)),
        "rf_shap": rf_shap_summary,
        "promotion": promotion,
    }
    write_json(summary_path, summary)
    return {
        "comparison": comparison,
        "predictions": predictions,
        "uncertainty": uncertainty,
        "promotion": promotion,
        "summary": summary,
    }


def _row_to_macro_metric(row: pd.Series, comparison: pd.DataFrame, target: str) -> dict[str, object]:
    year_row = comparison[(comparison["target"] == target) & (comparison["model"] == "year_only_xgb")]
    year_mae = float(year_row.iloc[0]["loso_mae_pp"]) if not year_row.empty else None
    year_r2 = float(year_row.iloc[0]["loso_r2"]) if not year_row.empty else None
    return {
        "loso_mae_pp": float(row["loso_mae_pp"]),
        "loso_r2": float(row["loso_r2"]),
        "loso_spearman": float(row["loso_spearman"]),
        "loso_pearson": float(row["loso_pearson"]),
        "n_state_years": int(row["n_state_years"]),
        "n_states": int(row["n_states"]),
        "n_survey_years": int(row["n_survey_years"]),
        "ablation_year_only_mae_pp": year_mae,
        "ablation_year_only_r2": year_r2,
        "satellite_contribution_pp": round(year_mae - float(row["loso_mae_pp"]), 4) if year_mae is not None else None,
        "selected_model": str(row["model"]),
        "feature_policy": str(row["feature_policy"]),
        "feature_count": int(row["feature_count"]),
    }


def _rf_continuity_note(comparison: pd.DataFrame, uncertainty: pd.DataFrame, promotion: dict[str, object]) -> str:
    rf_rows = comparison[comparison["model"].astype(str).str.startswith("rf_")].copy()
    xgb_rows = comparison[comparison["model"].isin(["current_xgb", "year_only_xgb"])].copy()
    selected_cols = [
        "target",
        "model",
        "feature_policy",
        "feature_count",
        "loso_mae_pp",
        "loso_rmse",
        "loso_r2",
        "loso_spearman",
    ]
    table = pd.concat([rf_rows, xgb_rows], ignore_index=True)
    table = table[selected_cols].sort_values(["target", "loso_mae_pp", "model"])
    rf_uncertainty = uncertainty[uncertainty["model"].astype(str).str.startswith("rf_")].copy()
    uncertainty_table = rf_uncertainty[
        ["target", "model", "metric", "observed", "ci_lower", "ci_upper", "n_boot", "n_groups"]
    ].sort_values(["target", "model", "metric"])
    return "\n".join(
        [
            "# Malaysia State Random Forest Continuity Benchmark",
            "",
            "Purpose: document the Random Forest benchmark and the deterministic state-level selection decision.",
            "",
            "Continuity facts:",
            "",
            "- Part 1 proposed Random Forest as the Semester 2 non-parametric modelling direction.",
            "- No retained executable old Random Forest modelling script is present in the final package.",
            "- Random Forest has therefore been reimplemented as a Malaysia state-level benchmark in `notebooks/03_model_training_evaluation.ipynb` inside this notebook.",
            "- Scope is state-level only. Only Malaysia state-level modelling forms part of the active project.",
            "- The primary model is selected by an encoded rule restricted to sensor-safe features; no manual override is used.",
            "",
            "Promotion status:",
            "",
            f"- Selected absolute-poverty point predictor: `{promotion['selected_model']}`.",
            f"- Best observed absolute-poverty model in this run: `{promotion['best_absolute_model']}`.",
            f"- Selection status: `{promotion['manual_promotion_review_status']}`.",
            "- RF is not claimed as decisively superior because the paired MAE uncertainty interval overlaps zero.",
            "- Relative poverty is best handled as year/trend-driven because the year-only baseline has the strongest state-level metrics.",
            "",
            "RF and state headline benchmark comparison:",
            "",
            _markdown_table(table),
            "",
            "RF uncertainty intervals:",
            "",
            _markdown_table(uncertainty_table),
            "",
        ]
    )


def _markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "_No rows._"
    rows = df.copy()
    for col in rows.columns:
        if pd.api.types.is_float_dtype(rows[col]):
            rows[col] = rows[col].map(lambda value: f"{value:.4f}")
        else:
            rows[col] = rows[col].astype(str)
    header = "| " + " | ".join(rows.columns) + " |"
    divider = "| " + " | ".join(["---"] * len(rows.columns)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows.to_numpy(dtype=str)]
    return "\n".join([header, divider, *body])



## Input Panel

The state panel remains the primary validated dataset: 16 states/federal territories, 10 DOSM survey years, and 158 state-year observations.

In [2]:
panel_path = PROJECT_ROOT / "data" / "state_year_panel_modelready_2002_2022.csv"
panel = pd.read_csv(panel_path)
input_summary = {
    "panel_path": str(panel_path.relative_to(PROJECT_ROOT)),
    "rows": int(len(panel)),
    "columns": int(len(panel.columns)),
    "states": int(panel["state_key"].nunique()),
    "years": sorted(int(year) for year in panel["year"].dropna().unique()),
    "targets": [col for col in ["poverty_absolute", "poverty_relative", "poverty_hardcore"] if col in panel.columns],
}
display(pd.DataFrame([input_summary]))
display(panel[["state", "year", "poverty_absolute", "poverty_relative", "wmean_ntl_mean", "wmean_ntl_mean_z_by_year", "wmean_ntl_mean_rank_by_year", "wmean_ntl_sensor_code"]].head(12))


,panel_path,rows,columns,states,years,targets
0,data\state_year_panel_modelready_2002_2022.csv,158,70,16,"[2002, 2004, 2007, 2009, 2012, 2014, 2016, 201...","[poverty_absolute, poverty_relative, poverty_h..."


,state,year,poverty_absolute,poverty_relative,wmean_ntl_mean,wmean_ntl_mean_z_by_year,wmean_ntl_mean_rank_by_year,wmean_ntl_sensor_code
0,Johor,2002,1.8,16.1,32.882334,-0.027503,0.6250,0.0
1,Johor,2004,2.0,15.3,41.218844,0.026470,0.6250,0.0
2,Johor,2007,1.5,14.2,40.370438,-0.027236,0.6250,0.0
3,Johor,2009,1.3,17.2,40.247728,0.014699,0.6250,0.0
4,Johor,2012,0.9,16.1,18.238911,0.085657,0.6875,1.0
5,Johor,2014,0.0,10.2,19.516877,0.094409,0.7500,1.0
6,Johor,2016,0.0,13.5,21.697619,0.161427,0.6875,1.0
7,Johor,2019,3.9,15.3,25.541486,0.394356,0.7500,1.0
8,Johor,2020,5.9,13.9,26.115878,0.389134,0.7500,1.0
9,Johor,2022,4.6,15.9,24.692999,0.380979,0.6875,1.0


## State Model Benchmark

Run the local training benchmark. Primary validation is leave-one-state-out. Uncertainty intervals bootstrap held-out states, not individual rows, so within-state residual dependence is preserved.

In [3]:
benchmark = run_state_benchmark(project_root=PROJECT_ROOT, n_boot=2000, random_state=42)
comparison = benchmark["comparison"]
uncertainty = benchmark["uncertainty"]
predictions = benchmark["predictions"]
promotion = benchmark["promotion"]
summary = benchmark["summary"]
print(json.dumps(promotion, indent=2, default=str))


C:\Users\farih\Documents\Codex\Portfolio\malaysia-poverty-monitor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "promoted": true,
  "selected_model": "rf_sensor_safe",
  "relative_selected_model": "year_only_xgb",
  "best_absolute_model": "rf_sensor_safe",
  "current_absolute_model": "current_xgb",
  "rule": "For absolute poverty, restrict eligibility to the predeclared sensor-safe feature policy, then select lowest LOSO MAE; break ties by higher Spearman correlation and model name. Relative poverty uses the lowest-MAE benchmark and remains a secondary negative result.",
  "manual_promotion_required": false,
  "manual_promotion_review_status": "not_applicable_deterministic_rule",
  "manual_promotion_reason": "No manual override is permitted by the encoded selection rule.",
  "current_absolute": {
    "target": "poverty_absolute",
    "model": "current_xgb",
    "feature_policy": "current_full",
    "feature_count": 20,
    "loso_mae_pp": 2.5372,
    "loso_rmse": 4.6639,
    "loso_r2": 0.1835,
    "loso_spearman": 0.8102,
    "loso_pearson": 0.4944,
    "n_state_years": 157,
    "n_states": 1

## Model Comparison

Lower MAE/RMSE is better; higher R2/Spearman/Pearson is better. The current XGBoost policy remains the benchmark to beat for the absolute-poverty state story.

In [4]:
display(comparison.sort_values(["target", "loso_mae_pp", "model"]))


,target,model,feature_policy,feature_count,loso_mae_pp,loso_rmse,loso_r2,loso_spearman,loso_pearson,n_state_years,n_states,n_survey_years
0,poverty_absolute,rf_sensor_safe,sensor_safe,11,2.4425,4.3923,0.2759,0.7945,0.5559,157,16,10
1,poverty_absolute,rf_current_full,current_full,20,2.5122,4.5445,0.2248,0.7886,0.5147,157,16,10
2,poverty_absolute,current_xgb,current_full,20,2.5372,4.6639,0.1835,0.8102,0.4944,157,16,10
3,poverty_absolute,sensor_safe_xgb,sensor_safe,11,2.6068,4.7364,0.1580,0.7851,0.4733,157,16,10
4,poverty_absolute,sensor_safe_year_dummies_ridge,sensor_safe_year_dummies,11,2.8177,4.6056,0.2038,0.7316,0.4951,157,16,10
5,poverty_absolute,year_only_xgb,year_only,1,2.9609,4.8456,0.1187,0.4910,0.4161,157,16,10
6,poverty_absolute,sensor_safe_elasticnet,sensor_safe,11,3.2968,5.0453,0.0445,0.5285,0.3255,157,16,10
7,poverty_absolute,sensor_safe_ridge,sensor_safe,11,3.6556,5.3334,-0.0677,0.4462,0.2932,157,16,10
8,poverty_relative,year_only_xgb,year_only,1,2.2364,2.8891,0.2472,0.4313,0.5112,158,16,10
9,poverty_relative,current_xgb,current_full,20,2.5228,3.2736,0.0335,0.4271,0.3882,158,16,10


## Uncertainty Bands

Intervals are fold/group bootstrap intervals over held-out states. They are intended for defensible reporting, not proof of causal effects.

In [5]:
selected_models = {
    "poverty_absolute": promotion["selected_model"],
    "poverty_relative": promotion["relative_selected_model"],
}
selected_uncertainty = uncertainty[
    uncertainty.apply(lambda row: row["model"] == selected_models.get(row["target"]), axis=1)
].sort_values(["target", "metric"])
display(selected_uncertainty)


,target,model,metric,observed,ci_lower,ci_upper,n_boot,group_col,n_groups
10,poverty_absolute,rf_sensor_safe,mae,2.442483,1.336837,3.847151,2000,state_key,16
14,poverty_absolute,rf_sensor_safe,pearson,0.555862,0.365211,0.849188,2000,state_key,16
12,poverty_absolute,rf_sensor_safe,r2,0.275861,-0.065536,0.660061,2000,state_key,16
11,poverty_absolute,rf_sensor_safe,rmse,4.392331,2.134657,6.412987,2000,state_key,16
13,poverty_absolute,rf_sensor_safe,spearman,0.794487,0.684893,0.876135,2000,state_key,16
75,poverty_relative,year_only_xgb,mae,2.236430,1.884216,2.628708,2000,state_key,16
79,poverty_relative,year_only_xgb,pearson,0.511206,0.396741,0.640698,2000,state_key,16
77,poverty_relative,year_only_xgb,r2,0.247220,0.089802,0.374573,2000,state_key,16
76,poverty_relative,year_only_xgb,rmse,2.889139,2.414878,3.361614,2000,state_key,16
78,poverty_relative,year_only_xgb,spearman,0.431267,0.309800,0.566573,2000,state_key,16


## Fold-Level Residual Evidence

The rows below show the worst held-out state-years for the selected absolute-poverty model. These are useful for explaining where the state-level model remains fragile.

In [6]:
abs_model = promotion["selected_model"]
abs_pred = predictions[(predictions["target"] == "poverty_absolute") & (predictions["model"] == abs_model)]
display(abs_pred.sort_values("abs_error", ascending=False).head(12))


,state_key,state,year,target,model,feature_policy,actual,predicted,residual,abs_error,feature_count
746,Sabah,Sabah,2004,poverty_absolute,rf_sensor_safe,sensor_safe,24.2,3.404470,-20.795530,20.795530,11
753,Sabah,Sabah,2020,poverty_absolute,rf_sensor_safe,sensor_safe,25.3,8.085433,-17.214567,17.214567,11
748,Sabah,Sabah,2009,poverty_absolute,rf_sensor_safe,sensor_safe,19.7,2.819363,-16.880637,16.880637,11
656,Kelantan,Kelantan,2020,poverty_absolute,rf_sensor_safe,sensor_safe,21.2,7.822255,-13.377745,13.377745,11
747,Sabah,Sabah,2007,poverty_absolute,rf_sensor_safe,sensor_safe,16.4,3.078854,-13.321146,13.321146,11
745,Sabah,Sabah,2002,poverty_absolute,rf_sensor_safe,sensor_safe,16.0,3.624553,-12.375447,12.375447,11
752,Sabah,Sabah,2019,poverty_absolute,rf_sensor_safe,sensor_safe,19.5,7.442343,-12.057657,12.057657,11
754,Sabah,Sabah,2022,poverty_absolute,rf_sensor_safe,sensor_safe,19.7,8.020608,-11.679392,11.679392,11
776,Trengganu,Terengganu,2004,poverty_absolute,rf_sensor_safe,sensor_safe,15.4,4.103648,-11.296352,11.296352,11
699,Pahang,Pahang,2007,poverty_absolute,rf_sensor_safe,sensor_safe,1.7,12.967835,11.267835,11.267835,11


## Report LOSO Figure

In [7]:
import matplotlib.pyplot as plt

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

rf_abs = predictions[
    (predictions["target"] == "poverty_absolute")
    & (predictions["model"] == promotion["selected_model"])
].copy()
state_mae = (
    rf_abs.groupby("state", as_index=False)["abs_error"]
    .mean()
    .sort_values("abs_error", ascending=False)
)
overall_mae = float(rf_abs["abs_error"].mean())

fig, ax = plt.subplots(figsize=(7.4, 4.8))
colors = ["#C00000" if val >= overall_mae else "#2E75B6" for val in state_mae["abs_error"]]
ax.barh(state_mae["state"], state_mae["abs_error"], color=colors)
ax.axvline(overall_mae, color="#333333", linestyle="--", linewidth=1.2, label=f"Mean MAE = {overall_mae:.2f} pp")
ax.invert_yaxis()
ax.set_title("RF State-Level LOSO Error by Held-Out State")
ax.set_xlabel("Mean absolute error (percentage points)")
ax.set_ylabel("Held-out state")
ax.grid(axis="x", alpha=0.2)
ax.legend(frameon=False, fontsize=8, loc="lower right")
for y, val in enumerate(state_mae["abs_error"]):
    ax.text(val + 0.08, y, f"{val:.2f}", va="center", fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig5_5_loso_by_state.png", dpi=180, bbox_inches="tight")
plt.close(fig)

pd.DataFrame({"figure": ["outputs/figures/fig5_5_loso_by_state.png"], "overall_mae": [overall_mae]})

,figure,overall_mae
0,outputs/figures/fig5_5_loso_by_state.png,2.442483


## Relative Poverty Negative Result


In [8]:
relative_comparison = comparison[comparison["target"] == "poverty_relative"].copy()
display(relative_comparison.sort_values(["loso_mae_pp", "model"]))
print("Relative poverty remains a secondary negative result: satellite features do not beat the year-only baseline.")


,target,model,feature_policy,feature_count,loso_mae_pp,loso_rmse,loso_r2,loso_spearman,loso_pearson,n_state_years,n_states,n_survey_years
8,poverty_relative,year_only_xgb,year_only,1,2.2364,2.8891,0.2472,0.4313,0.5112,158,16,10
9,poverty_relative,current_xgb,current_full,20,2.5228,3.2736,0.0335,0.4271,0.3882,158,16,10
10,poverty_relative,rf_current_full,current_full,20,2.6302,3.4041,-0.0451,0.3654,0.3296,158,16,10
11,poverty_relative,rf_sensor_safe,sensor_safe,11,2.6314,3.4699,-0.0859,0.3445,0.3090,158,16,10
12,poverty_relative,sensor_safe_xgb,sensor_safe,11,2.6384,3.4618,-0.0807,0.3561,0.3228,158,16,10
13,poverty_relative,sensor_safe_elasticnet,sensor_safe,11,2.8150,3.5450,-0.1334,0.2300,0.2230,158,16,10
14,poverty_relative,sensor_safe_year_dummies_ridge,sensor_safe_year_dummies,11,3.8229,4.8174,-1.0930,0.0741,0.0738,158,16,10
15,poverty_relative,sensor_safe_ridge,sensor_safe,11,3.8369,4.8099,-1.0864,0.0800,0.0895,158,16,10


Relative poverty remains a secondary negative result: satellite features do not beat the year-only baseline.


## Certification Summary

The dashboard contract is refreshed only from locally regenerated artifacts. Geographic LOSO scores are the headline evidence; rolling-origin results are reported separately as a stricter future-year robustness check. Neither evaluation supports causal or official-statistics claims.


## NTL Rank-Shift Verification

Verify that z-score normalisation preserves rank ordering across the 2009-2012 DMSP-to-VIIRS sensor boundary.
Runtime result: 16 of 16 states remain within 1 rank position (as documented in the FYP report methodology section).

In [9]:
# NTL rank-shift verification across 2009->2012 sensor boundary
# NTL rank-shift verification: runtime result 16/16 states within 1 rank position (see FYP report Chapter 4, SS4.3.2)

rank_col = "wmean_ntl_mean_rank_by_year"
if rank_col not in panel.columns:
    print(f"WARNING: {rank_col!r} not in panel â€” skipping rank-shift verification")
    ntl_rank_shift = {"skipped": True, "reason": f"{rank_col} not found"}
else:
    panel_2009 = panel[panel["year"] == 2009][["state_key", rank_col]].set_index("state_key")
    panel_2012 = panel[panel["year"] == 2012][["state_key", rank_col]].set_index("state_key")
    common_states = panel_2009.index.intersection(panel_2012.index)
    shift = (panel_2012.loc[common_states, rank_col] - panel_2009.loc[common_states, rank_col]).abs()
    per_state = {str(k): round(float(v), 3) for k, v in shift.items()}
    within_1 = int((shift < 1).sum())
    outside_1 = int((shift >= 1).sum())
    claim_verified = within_1 >= 14
    ntl_rank_shift = {
        "claim": "rank shifts < 1 position for 16/16 states at 2009->2012 sensor boundary (runtime verified)",
        "metric_used": rank_col,
        "boundary_year_pair": [2009, 2012],
        "states_compared": len(common_states),
        "per_state_rank_shift": per_state,
        "states_within_1_position": within_1,
        "states_outside_1_position": outside_1,
        "claim_verified": claim_verified,
        "notes": (
            "Rank is computed per survey year across all states. A shift of < 1 position means "
            "the state did not swap ordinal position relative to its neighbours at the sensor boundary. "
            "z-score normalization is used because it preserves this rank ordering without requiring "
            "a polynomial cross-sensor fit."
        ),
    }
    write_json(QA_DIR / "ntl_rank_shift_verification.json", ntl_rank_shift)
    print(f"Claim verified: {claim_verified}")
    print(f"States within 1 rank position: {within_1}/16")
    print(f"States outside 1 rank position: {outside_1}/16")
    display(shift.sort_values(ascending=False).rename("abs_rank_shift").to_frame())

Claim verified: True
States within 1 rank position: 16/16
States outside 1 rank position: 0/16


,abs_rank_shift
state_key,
Labuan,0.1875
Trengganu,0.1875
Johor,0.0625
Kedah,0.0625
Kelantan,0.0625
NegeriSembilan,0.0625
Pahang,0.0625
Perak,0.0625
Sabah,0.0625


In [10]:
metrics = read_json(METRICS_DIR / "malaysia_state_selected_model_loso_metrics.json")
metric_targets = metrics.get("targets", {})
notebook_summary = {
    "dashboard_prediction_artifact_rows": int(len(pd.read_csv(PROJECT_ROOT / "data" / "state_model_predictions.csv"))),
    "state_count": int(panel["state_key"].nunique()),
    "years": sorted(int(year) for year in panel["year"].dropna().unique()),
    "targets": metric_targets,
    "benchmark_summary_path": "outputs/qa/malaysia_state_benchmark_summary.json",
    "model_comparison_path": "outputs/metrics/malaysia_state_model_comparison.csv",
    "uncertainty_path": "outputs/metrics/malaysia_state_uncertainty_intervals.csv",
    "temporal_backtest_path": "outputs/metrics/malaysia_state_temporal_backtest.csv",
    "temporal_backtest_summary_path": "outputs/qa/malaysia_state_temporal_backtest.json",
    "promotion_path": "outputs/qa/malaysia_state_model_promotion.json",
    "promotion": promotion,
    "claim_contract": "Validated Malaysia state-level monitoring branch only.",
}
write_json(QA_DIR / "malaysia_model_training_notebook_summary.json", notebook_summary)
display(pd.DataFrame(metric_targets).T)


,loso_mae_pp,loso_r2,loso_spearman,loso_pearson,n_state_years,n_states,n_survey_years,ablation_year_only_mae_pp,ablation_year_only_r2,satellite_contribution_pp,selected_model,feature_policy,feature_count
poverty_absolute,2.4425,0.2759,0.7945,0.5559,157,16,10,2.9609,0.1187,0.5184,rf_sensor_safe,sensor_safe,11
poverty_relative,2.2364,0.2472,0.4313,0.5112,158,16,10,2.2364,0.2472,0.0,year_only_xgb,year_only,1
